### Check_relevance 유사도 체크 (node 1-2)

##### 담당 범위
- **NODE 1**: Moderator 1 — 정규식 기반 민감정보 감지
- **NODE 2**: Moderator 2 — LLM 기반 질문 의도 파악 + 메타데이터 필터 선정 + 확답/법률자문 거르기

##### 서비스 원칙
본 서비스는 법령·판례·해석례에 기반한 **정보 안내**만 제공합니다.  
법률 자문, 법적 조언, 확정적 판단은 제공하지 않으며 모든 답변에 이를 고지합니다.

In [1]:
# 파이썬 실행 환경 제어 모듈 불러오기
import sys
sys.path.insert(0, '..')

In [4]:
import os
from dotenv import load_dotenv

load_dotenv('../.env')


True

---
### 공통 State 정의
팀 전체 노드가 공유하는 State (팀원 협의 기준)

In [ ]:
from typing import TypedDict, Annotated, Optional
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage


class ModeratorState(TypedDict):
    messages:         Annotated[list[BaseMessage], add_messages]   # 전체
    user_input:       str                 # 현재 사용자 입력 원문 (초기값)
    intent_metadata:  Optional[str]       # RAG 검색 필터 (moderator2 설정)
    retrieved_docs:   list                # 검색된 문서 목록 (retrieval 설정)
    similarity_score: Optional[float]     # 검색 최고 유사도 (retrieval 설정)
    is_sensitive:     bool                # 민감정보 포함 여부 (moderator1 설정)
    is_definitive:    bool                # 확답 요구 질문 여부 (moderator2 설정)
    needs_link:       bool                # 링크 요청 여부 - Tavily 트리거 (moderator2 설정)
    final_answer:     Optional[str]       # 최종 답변 (generator~moderator3 설정)
    


### NODE 1: Moderator 1 — 정규식 기반 민감정보 감지

LLM 없이 정규식만으로 처리 → 빠르고 비용 없음

감지 대상:
- 주민등록번호
- 계좌번호
- 전화번호 (선택적 포함 가능)

In [5]:
# 사용자 input - 민감정보 정규식 패턴 정의 (NODE 1)
import re

SENSITIVE_PATTERNS = {
    '주민등록번호': re.compile(
        r'(?<!\d)\d{6}[-\s]?[1-4]\d{6}(?!\d)'
    ),
    '계좌번호': re.compile(
        r'(?<!\d)\d{3,6}[-\s]\d{2,6}[-\s]\d{2,6}(?!\d)'
        # [-\s]? → [-\s] 로 변경 (하이픈 필수로 강제, 오탐 방지)
    ),
    '전화번호': re.compile(
        r'(?<!\d)(010|011|016|017|018|019)[-\s]?\d{3,4}[-\s]?\d{4}(?!\d)'
    ),
}
# \d{6} -> 숫자 6개 (생년월일)
# [-\s]? '-' 또는 공백, 있어도 되고 없어도 됨
# [1-4] -> 1~4 중 하나 (성별 코드)
# \d{6} -> 숫자 6개 
# \b -> 단어 경계 (숫자가 더 이어지면 매칭x)

# Fallback 메시지 (차단되었을 때 유저에게 보여줄 메시지)
SENSITIVE_FALLBACK = (
    "민감한 정보(주민등록번호, 계좌번호 등)가 포함되어 있어 처리할 수 없어요.\n"
    "개인정보를 제외하고 다시 질문해 주세요! 😊"
)


def detect_sensitive(text: str) -> tuple[bool, list[str]]:
    """
    민감정보 감지 함수

    Returns:
        (감지 여부, 감지된 항목 목록)
    """
    detected = []
    for name, pattern in SENSITIVE_PATTERNS.items():
        if pattern.search(text):        # 텍스트 전체에서 패턴 찾기
            detected.append(name)
    return len(detected) > 0, detected   # 두개 동시 반환 (예시: (True, ['주민등록번호']))



In [6]:
# NODE 1 단위 테스트 

test_inputs = [
    # (입력, 감지 예상)
    ("보증금 반환 기한이 어떻게 되나요?",               False),  # 정상
    ("제 주민번호는 901234-1234567 입니다",              True),   # 주민번호
    ("계좌번호 110-123-456789로 보내주세요",             True),   # 계좌번호
    ("010-1234-5678로 연락주세요",                       True),   # 전화번호
    ("계약갱신청구권을 행사하려면 어떻게 해야 하나요?",   False),  # 정상
    ("주민번호 없이 대항력 취득이 가능한가요?",           False),  # '주민번호' 단어만, 번호 없음
]

print('--- NODE 1 단위 테스트 ---')
all_pass = True
for text, expected in test_inputs:
    detected, items = detect_sensitive(text)
    status = 'PASS' if detected == expected else 'FAIL'
    if detected != expected:
        all_pass = False
    print(f'{status} | 감지: {detected} (예상: {expected}) | 항목: {items}')
    print(f'     입력: {text[:50]}')
    print()

print(f'결과: {"전체 통과" if all_pass else "일부 실패 → 정규식 조정 필요"}')

--- NODE 1 단위 테스트 ---
PASS | 감지: False (예상: False) | 항목: []
     입력: 보증금 반환 기한이 어떻게 되나요?

PASS | 감지: True (예상: True) | 항목: ['주민등록번호']
     입력: 제 주민번호는 901234-1234567 입니다

PASS | 감지: True (예상: True) | 항목: ['계좌번호']
     입력: 계좌번호 110-123-456789로 보내주세요

PASS | 감지: True (예상: True) | 항목: ['계좌번호', '전화번호']
     입력: 010-1234-5678로 연락주세요

PASS | 감지: False (예상: False) | 항목: []
     입력: 계약갱신청구권을 행사하려면 어떻게 해야 하나요?

PASS | 감지: False (예상: False) | 항목: []
     입력: 주민번호 없이 대항력 취득이 가능한가요?

결과: 전체 통과


In [7]:
# LangGraph NODE 1 함수 
from langgraph.graph import MessagesState 
from langchain_core.messages import AIMessage


def moderator_1(state: MessagesState) -> dict:      # ← MessagesState → ModeratorState
    """
    NODE 1: 정규식 기반 민감정보 감지

    감지됨  → Fallback 메시지 추가, is_sensitive=True → END로 라우팅
    정상    → is_sensitive=False, messages 유지 → NODE 2로
    """
    user_input = state['messages'][-1].content
    detected, items = detect_sensitive(user_input)

    if detected:
        return {
            'messages': [AIMessage(content=SENSITIVE_FALLBACK)],
            'is_sensitive': True,
        }

    return {
        # 'messages': state['messages'], -> messages 키 완전히 제거 (add_messages reducer가 중복 추가 방지)
        'is_sensitive': False,
    }
# messages 키 빼버림으로써 -> messages는 건드리지 않고 is_sensitive만 업데이트됨
# -> 다음 노드 (moderator_2로 기존 messages가 그대로 전달됨)


### NODE 2: Moderator 2 — LLM 기반 사용자 질문 의도 파악 + 메타데이터 필터 선정 + 확답/법률자문 거르기


우리 프로젝트 메타데이터:
- `source`: eflaw(법령) / prec(판례) / expc(해석례)
- `법령명`: 주택임대차보호법, 민법 등
- `조문번호`: 제3조, 제6조의3 등

서비스 원칙: 법률 자문/법적 조언/확정적 판단 요청은 모두 차단 후 고지

In [8]:
# 확답 질문 판별 프롬프트 
# NODE 2 LLM 체인 정의


from pydantic import BaseModel, Field
from typing import Literal
from langchain_core.prompts import PromptTemplate
from langchain.chat_models import init_chat_model
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
# llm모델 추후 선정 시 입력

# 확답 질문 판별 
class IntentResult(BaseModel):
    is_definitive: str = Field(
        description=("'yes' if 아래 중 하나에 해당:\n"
            "  - 확답/확정적 판단 요구 ('무조건', '100% 맞죠?' 등)\n"
            "  - 법률 자문 요청 ('법적으로 어떻게 해야 하나요?')\n"
            "  - 법적 조언 요청 ('저는 어떻게 하면 좋을까요?')\n"
            "'no' if 법령/판례/해석 정보를 조회하는 질문"
        )
    )
    metadata_filter: str = Field(
        description="검색할 데이터 소스: eflaw, prec, expc, all 중 하나"
    )
    needs_link: str = Field(
        description="'yes' if 사용자가 관련 자료 링크/출처를 요청함, 'no' otherwise"
    )
    reason: str = Field(
        description="판단 이유 중 한줄 요약"
    )


MODERATOR_2_PROMPT = PromptTemplate.from_template("""
당신은 임대차 계약 법령 정보 안내 서비스의 질문 분류기입니다.
                                                  
[서비스 원칙]
본 서비스는 법령·판례·해석례에 기반한 정보 안내만 제공합니다.
법률 자문, 법적 조언, 확정적 판단은 제공하지 않습니다.
                                                  
아래 질문을 분석하여 다음을 판단하세요:

1. **is_definitive: 법률 자문/조언/확답 요청 여부**
   다음 중 하나라도 해당하면 'yes':
   - 확정적 판단 요구: '무조건 됩니까?', '100% 맞죠?', '확실히 이기나요?'
   - 법률 자문 요청: '법적으로 어떻게 해야 하나요?', '어떻게 하면 될까요?'
   - 법적 조언 요청: '저는 어떤 조치를 취해야 하나요?', '무엇을 해야 하나요?'
   단순 정보 조회('~기한이 어떻게 되나요?', '~요건이 무엇인가요?')는 'no'

2. **metadata_filter: 검색할 데이터 소스 선정**
   - eflaw: 법령 조문이 필요한 질문 (법 조항, 기간, 요건 등)
   - prec: 판례가 필요한 질문 (분쟁, 소송, 판결 결과 등)
   - expc: 법령 해석이 필요한 질문 (애매한 조항 해석, 예외 상황 등)
   - all: 모두 필요하거나 판단 불가

3. **needs_link: 관련 자료 링크 요청 여부**
   - 'yes': '링크', '자료', '참고', '출처', 'URL' 등 링크/출처를 요청한 경우
   - 'no': 그 외

질문: {question}
""")

intent_chain = MODERATOR_2_PROMPT | llm.with_structured_output(IntentResult)


OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

In [ ]:
# Fallback 메시지 

DEFINITIVE_FALLBACK = (
    "⚠️ 본 서비스는 법률 자문 또는 법적 조언을 제공하지 않습니다.\n\n"
    "저희는 주택임대차 관련 법령·판례·해석례에 기반한 정보 안내만 제공하며,\n"
    "확정적인 법률 판단은 드리기 어렵습니다.\n\n"
    "구체적인 법적 판단이 필요하신 경우, 법률 전문가 또는 법률구조공단과 상담하시길 권해드립니다."
)


In [ ]:
# NODE 2 단위 테스트 

test_questions = [
    # (질문, 확답 예상, metadata_filter 예상)
    ("보증금 반환 기한이 어떻게 되나요?",                    False, "eflaw"),
    ("제가 이 경우 무조건 이길 수 있죠?",                    True,  "all"),
    ("임대인이 보증금을 안 돌려줬을 때 판례가 있나요?",       False, "prec"),
    ("계약갱신청구권 거절 사유가 애매한데 어떻게 해석하나요?", False, "expc"),
    ("법적으로 저는 어떻게 해야 하나요?",                      True,  "all"), # 법적 조언 요청
]

print('--NODE 2 단위 테스트--')

for question, expect_def, expect_src in test_questions:
    result: IntentResult = intent_chain.invoke({'question': question})
    is_def = result.is_definitive == 'yes'
    def_ok = '✅' if is_def == expect_def else '❌'
    src_ok = '✅' if result.source_filter == expect_src else '⚠️'

    print(f'Q: {question}')
    print(f'  확답/자문: {def_ok} {result.is_definitive} (예상: {"yes" if expect_def else "no"})')
    print(f'  metadata_filter: {src_ok} {result.metadata_filter} (예상: {expect_src})')
    print(f'  needs_link: {result.needs_link}')
    print(f'  이유: {result.reason}')
    print()

# llm 모델 입력되어있지 않아서 에러남

--NODE 2 단위 테스트--


TypeError: _init_chat_model_helper() missing 1 required positional argument: 'model'

In [ ]:
# LangGraph NODE 2 함수


def moderator_2(state: ModeratorState) -> dict:  # ← MessagesState → ModeratorState
    """
    NODE 2: LLM 기반 질문 의도 파악

    확답/법률자문 질문  → Fallback 메시지 후, is_definitive=True -> END
    정상 질문  → metadata_filter, needs_link 설정 → retrieval로
    """
    question = state['messages'][-1].content
    result: IntentResult = intent_chain.invoke({'question': question})

    if result.is_definitive == 'yes':
        return {
            'messages': [AIMessage(content=DEFINITIVE_FALLBACK)],
            'is_definitive': True,

        }
    # messages 키 skip (add_messages reducer 중복 방지)
    return {
        # 'messages': state['messages'],
        'is_definitive': False,
        'metadata_filter': result.metadata_filter,
        'needs_link': result.needs_link == 'yes',
    }



### LangGraph 연결: NODE 1 → NODE 2 파이프라인


In [ ]:
# 라우팅 함수 

from langgraph.graph import StateGraph, START, END
from typing import Literal


def route_after_mod1(state: ModeratorState) -> Literal['moderator_2', END]:
    """NODE 1 결과에 따라 라우팅"""
    if state.get('is_sensitive'):
        return END
    return 'moderator_2'


def route_after_mod2(state: ModeratorState) -> Literal['retrieval', END]:
    """NODE 2 결과에 따라 라우팅"""
    if state.get('is_definitive'):
        return END
    return 'retrieval'  # 다음 팀원 노드



In [ ]:
# 임시 retrieval 노드 (팀원 연결 전 stub) 

def retrieval_stub(state: ModeratorState) -> dict:
    """팀원 retrieval 노드 연결 전 임시 stub"""
    source = state.get('source_filter', 'all')
    return {
        'messages': [AIMessage(
            content=f'[retrieval stub] metadata_filter={source} 로 검색 예정'
        )]
    }


In [ ]:
# 그래프 조립 
from langgraph.graph import StateGraph, START, END 
from langgraph.prebuilt import ToolNode, tools_condition 

builder = StateGraph(ModeratorState)

# 노드 등록
builder.add_node('moderator_1', moderator_1)
builder.add_node('moderator_2', moderator_2)
builder.add_node('retrieval',   retrieval_stub)

# 엣지 연결
builder.add_edge(START, 'moderator_1')
builder.add_conditional_edges('moderator_1', route_after_mod1)
builder.add_conditional_edges('moderator_2', route_after_mod2)
builder.add_edge('retrieval', END)

graph = builder.compile()
graph

In [ ]:
# 통합 테스트 

integration_cases = [
    # (케이스 설명, 입력)
    ("정상 질문",       "보증금 반환 기한이 어떻게 되나요?"),
    ("민감정보 포함",   "주민번호 901234-1234567 알려드릴게요"),
    ("확답 요청",       "제가 이 경우 무조건 이기는 거 맞죠?"),
    ("법적 조언 요청",  "저는 법적으로 어떻게 해야 하나요?"),
    ("판례 필요 질문",  "임대인이 보증금 안 줬을 때 판례 있나요?"),
    ("링크 요청 포함",  "계약갱신청구권 관련 자료 링크도 알려주세요"),
]

# 초기 State (팀 공통 필드 전체)
INITIAL_STATE = {
    'user_input':       '',
    'metadata_filter':  None,
    'retrieved_docs':   [],
    'similarity_score': None,
    'is_sensitive':     False,
    'is_definitive':    False,
    'needs_link':       False,
    'final_answer':     None,
}

for desc, question in integration_cases:
    print(f'\n{"="*50}')
    print(f'[{desc}] Q: {question}')
    print('='*50)

    init = {**INITIAL_STATE, 'messages': [('human', question)], 'user_input': question}

    for chunk in graph.stream(init):
        for node_name, update in chunk.items():
            print(f'  [{node_name}]')
            if 'messages' in update and update['messages']:
                last = update['messages'][-1]
                print(f'  → {last.content[:120]}')
            if update.get('is_sensitive'):
                print(f' 🚫 차단: 민감정보 감지')
            if update.get('is_definitive'):
                print(f' 🚫 차단: 확답/법률자문 요청 감지')
            if update.get('metadata_filter'):
                print(f' 🔍 metadata_filter: {update["metadata_filter"]}')
            if update.get('needs_link'):
                print(f' 🔗 needs_link: {update["needs_link"]}')

---
### 체크리스트 + 팀원 연결 가이드

테스트 결과 체크 리스트

##### NODE 1 확인
- [ ] 주민번호 포함 → Fallback 출력, END
- [ ] 계좌번호 포함 → Fallback 출력, END
- [ ] 전화번호 포함 → Fallback 출력, END
- [ ] 정상 질문 → NODE 2로 통과
- [ ] '주민번호'라는 단어만 있고 번호 없음 → 정상 통과

##### NODE 2 확인
- [ ] 확답 요청 질문 → Fallback 출력, END
- [ ] 법률 자문/조언 요청 → Fallback 출력, END
- [ ] 법령 조문 질문 → metadata_filter = eflaw
- [ ] 판례 질문 → metadata_filter = prec
- [ ] 해석 질문 → metadata_filter = expc
- [ ] 링크 요청 → needs_link = True
- [ ] 정상 질문 → retrieval로 통과 + metadata_filter 전달

---
### 팀원 연결 시 수정할 부분

1. `retrieval_stub` → 팀원 retrieval 노드 함수로 교체
2. `ModeratorState`는 팀 공통 State이므로 공유 모듈로 분리 예정
3. `metadata_filter`를 retrieval 노드에서 Qdrant 필터로 활용:
```python
# store.search() 호출 시 source 필터 적용 예시
# metadata_filter != 'all' 일 때만 필터 적용
filter_condition = {'source': metadata_filter} if metadata_filter != 'all' else None
results = store.search(query=question, top_k=5, filter=filter_condition)
```
4. 모든 최종 답변에 법률 자문 고지 문구 추가 (formatter / moderator3 담당):
** 본 서비스는 법률 자문 또는 법적 조언이나 법적인 판단을 제공하지 않습니다.